# Fashion-MNIST CNN

IMSA coursework publication copy. Saved outputs were cleared; rerun before citing results.


In [ ]:
from tensorflow import keras
from tensorflow.keras.datasets import fashion_mnist
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense
from tensorflow.keras.optimizers import Adam
from sklearn.metrics import confusion_matrix
import matplotlib.pyplot as plt
import numpy as np

## 1. Data Exploration

In [ ]:
(x_train, y_train), (x_test, y_test) = fashion_mnist.load_data()
print(f"Training set: {x_train.shape}")
print(f"Training labels: {y_train.shape}")
print(f"Test set: {x_test.shape}")
print(f"Test labels: {y_test.shape}")

In [ ]:
unique, counts = np.unique(y_train, return_counts=True)
class_names = [
    'T-shirt/top', 'Trouser', 'Pullover', 'Dress', 'Coat', 
    'Sandal', 'Shirt', 'Sneaker', 'Bag', 'Ankle boot'
]
for i, class_label in enumerate(unique):
    idx = np.where(y_train == class_label)[0][0]
    plt.imshow(x_train[idx], cmap='gray')
    plt.title(f"Label: {class_names[class_label]}")
    plt.show()


## 2. Data Preprocessing

In [ ]:
# Normalize reshape
x_train_normalized = x_train.astype('float32') / 255.0
x_test_normalized = x_test.astype('float32') / 255.0

x_train_reshaped = x_train_normalized.reshape(-1, 28, 28, 1)
x_test_reshaped = x_test_normalized.reshape(-1, 28, 28, 1)

print(f"Reshaped training data shape: {x_train_reshaped.shape}")
print(f"Reshaped test data shape: {x_test_reshaped.shape}")

# Split
from sklearn.model_selection import train_test_split
x_train_split, x_val, y_train_split, y_val = train_test_split(
    x_train_reshaped, y_train, test_size=0.2, random_state=42, stratify=y_train
)

print(f"Training set shape: {x_train_split.shape}")
print(f"Validation set shape: {x_val.shape}")
print(f"Test set shape: {x_test_reshaped.shape}")

## 3. Build the CNN Model

In [ ]:
model = Sequential([
    Conv2D(32, (3, 3), activation='relu', input_shape=(28, 28, 1)),
    MaxPooling2D((2, 2)),
    Conv2D(64, (3, 3), activation='relu'),
    MaxPooling2D((2, 2)),
    Flatten(),
    Dense(32, activation='relu'),
    Dense(10, activation='softmax')
])

# Compile the model
model.compile(
    loss='sparse_categorical_crossentropy',
    optimizer=Adam(learning_rate=0.01),
    metrics=['accuracy']
)

# Print model summary
model.summary()

## 4. Train the Model & Plot Learning Curves

In [ ]:
# Train the model
history = model.fit(
    x_train_split, y_train_split,
    epochs=5,
    batch_size=32,
    validation_data=(x_val, y_val),
    verbose=1
)

In [ ]:
# Plot learning curves
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 5))

# Training vs Validation Accuracy
ax1.plot(history.history['accuracy'], label='Training Accuracy')
ax1.plot(history.history['val_accuracy'], label='Validation Accuracy')
ax1.set_xlabel('Epochs')
ax1.set_ylabel('Accuracy')
ax1.set_title('Training vs Validation Accuracy')
ax1.legend()
ax1.grid(True)

# Training vs Validation Loss
ax2.plot(history.history['loss'], label='Training Loss')
ax2.plot(history.history['val_loss'], label='Validation Loss')
ax2.set_xlabel('Epochs')
ax2.set_ylabel('Loss')
ax2.set_title('Training vs Validation Loss')
ax2.legend()
ax2.grid(True)

plt.show()

## 5. Evaluate the Model

In [ ]:
# Evaluate the model on the test set
test_loss, test_accuracy = model.evaluate(x_test_reshaped, y_test, verbose=0)
print(f"Test Loss: {test_loss:.4f}")
print(f"Test Accuracy: {test_accuracy:.4f}")

# Determine if model is overfitting, underfitting, or well-fit
final_train_acc = history.history['accuracy'][-1]
final_val_acc = history.history['val_accuracy'][-1]

print(f"\nFinal Training Accuracy: {final_train_acc:.4f}")
print(f"Final Validation Accuracy: {final_val_acc:.4f}")
print(f"Final Test accuracy: {test_accuracy:.4f}")

# Analysis
print("The model appears to be well fit due to the training and test accuracy" \
" being close to each other (~0.01 difference).")



In [ ]:
# Visualize layers
from tensorflow.keras.models import Model

convolution_layer_outputs = [layer.output for layer in model.layers if 'conv2d' in layer.name]
convolution_layer_names = [layer.name for layer in model.layers if 'conv2d' in layer.name]

model_input = model.layers[0].input
activation_model = Model(inputs=model_input, outputs=convolution_layer_outputs)

test_image = x_test_reshaped[0:1]

activations = activation_model.predict(test_image)

# Visualize feature maps 
for layer_name, activation in zip(convolution_layer_names, activations):
    print(f"\n{layer_name} - Output shape: {activation.shape}")
    n_features = activation.shape[-1]
    
    # 8 channels per row
    n_cols = min(8, n_features)
    n_rows = (n_features + 7) // 8
    
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(14, n_rows * 2))
    
    axes_list = axes.flatten()
    
    # Go through all features
    for i in range(n_features):
        ax = axes_list[i]
        ax.imshow(activation[0, :, :, i], cmap='viridis')
        ax.set_title(f'Channel {i}')
        ax.axis('off')

    
    plt.suptitle(f'{layer_name} - Feature Maps', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.show()

## 6. Confusion Matrix & Error Analysis

In [ ]:
import seaborn as sns
# Create confusion matrix
class_names = [
    'T-shirt/top', 'Trouser', 'Pullover', 'Dress', 'Coat', 
    'Sandal', 'Shirt', 'Sneaker', 'Bag', 'Ankle boot'
]

y_pred = model.predict(x_test_reshaped)
y_pred_labels = np.argmax(y_pred, axis=1)

cm = confusion_matrix(y_test, y_pred_labels)

# Plot confusion matrix
plt.figure(figsize=(8, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Reds', 
            xticklabels=class_names, yticklabels=class_names)

plt.xlabel('Predicted Label')
plt.ylabel('True Label')
plt.show()


In [ ]:
# Find 5 misclassified images
misclassified_indices = np.where(y_test != y_pred_labels)[0]
print(f"Total misclassified images: {len(misclassified_indices)}")

fig, axes = plt.subplots(1, 5, figsize=(15, 3))

for i in range(5):
    idx = misclassified_indices[i]
    axes[i].imshow(x_test[idx], cmap='gray')
    true_label = y_test[idx]
    pred_label = y_pred_labels[idx]
    axes[i].set_title(f"True: {class_names[true_label]}\nPred: {class_names[pred_label]}") 
plt.show()


## Short Answer Questions

# 1. Why do we normalize image pixel values before training a neural network?
We normalilze to keep the input ranges of models small, allowing values in weights and inputs from becoming far too large in training.
# 2. Why do CNNs usually perform better than fully connected networks on image data?
CNN's use sliding kernels that can learn features on an image to then lead down to better classifications of the images regardless of the positioning of objects in the iamge.
# 3. What does a convolutional filter learn?
The filter learns to detect visual patterns in images like edges and corners, and eventaully complex shapes.